### Imports and MLflow setup

In [ ]:
import os
import sys
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from feast import FeatureStore
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from mlflow.tracking import MlflowClient

sys.path.insert(0, os.path.abspath("../../src"))
from utils.notify import notify_run_complete, notify_model_registered, notify_stage_change

_db_path = os.path.abspath("../../mlruns/mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
mlflow.set_experiment("employee-attrition")

SLACK_URL = os.getenv("SLACK_WEBHOOK_URL", "")  # paste your Slack Incoming Webhook URL here
MODEL_NAME = "employee-attrition"


### Load data + simple split

In [3]:
if os.path.exists("../../data/processed/employee_attrition.parquet"):
    employee_df = pd.read_parquet("../../data/processed/employee_attrition.parquet")
    print(f"Loaded {len(employee_df)} rows")
else:
    raise FileNotFoundError(
        "../../data/processed/employee_attrition.parquet not found — run prepare_feast_data.py first"
    )


Loaded 74498 rows


In [12]:
employee_df

,employee_id,age,gender,years_at_company,job_role,monthly_income,work-life_balance,job_satisfaction,performance_rating,number_of_promotions,...,job_level,company_size,company_tenure,remote_work,leadership_opportunities,innovation_opportunities,company_reputation,employee_recognition,attrition,event_timestamp
0,8410,31,Male,19,Education,5390,Excellent,Medium,Average,2,...,Mid,Medium,89,No,No,No,Excellent,Medium,Stayed,2026-06-12 17:30:05.567898
1,64756,59,Female,4,Media,5534,Poor,High,Low,3,...,Mid,Medium,21,No,No,No,Fair,Low,Stayed,2026-06-12 17:30:05.567898
2,30257,24,Female,10,Healthcare,8159,Good,High,Low,0,...,Mid,Medium,74,No,No,No,Poor,Low,Stayed,2026-06-12 17:30:05.567898
3,65791,36,Female,7,Education,3989,Good,High,High,1,...,Mid,Small,50,Yes,No,No,Good,Medium,Stayed,2026-06-12 17:30:05.567898
4,65026,56,Male,41,Education,4821,Fair,Very High,Average,0,...,Senior,Medium,68,No,No,No,Fair,Medium,Stayed,2026-06-12 17:30:05.567898
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74493,16243,56,Female,42,Healthcare,7830,Poor,Medium,Average,0,...,Senior,Medium,60,No,No,No,Poor,Medium,Stayed,2026-06-12 17:30:05.567898
74494,47175,30,Female,15,Education,3856,Good,Medium,Average,2,...,Entry,Medium,20,No,No,No,Good,Medium,Left,2026-06-12 17:30:05.567898
74495,12409,52,Male,5,Education,5654,Good,Very High,Below Average,0,...,Mid,Small,7,No,No,No,Good,High,Left,2026-06-12 17:30:05.567898
74496,9554,18,Male,4,Education,5276,Fair,High,Average,0,...,Mid,Large,5,No,No,No,Poor,High,Stayed,2026-06-12 17:30:05.567898


In [4]:
employee_df_encoded = employee_df.copy()
label_encoders = {}

for col in employee_df_encoded.select_dtypes(include=["object", "category"]).columns:
    le = LabelEncoder()
    employee_df_encoded[col] = le.fit_transform(employee_df_encoded[col])
    label_encoders[col] = le


In [15]:
employee_df_encoded

,employee_id,age,gender,years_at_company,job_role,monthly_income,work-life_balance,job_satisfaction,performance_rating,number_of_promotions,...,job_level,company_size,company_tenure,remote_work,leadership_opportunities,innovation_opportunities,company_reputation,employee_recognition,attrition,event_timestamp
0,8410,31,1,19,0,5390,0,2,0,2,...,1,1,89,0,0,0,0,2,1,2026-06-12 17:30:05.567898
1,64756,59,0,4,3,5534,3,0,3,3,...,1,1,21,0,0,0,1,1,1,2026-06-12 17:30:05.567898
2,30257,24,0,10,2,8159,2,0,3,0,...,1,1,74,0,0,0,3,1,1,2026-06-12 17:30:05.567898
3,65791,36,0,7,0,3989,2,0,2,1,...,1,2,50,1,0,0,2,2,1,2026-06-12 17:30:05.567898
4,65026,56,1,41,0,4821,1,3,0,0,...,2,1,68,0,0,0,1,2,1,2026-06-12 17:30:05.567898
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74493,16243,56,0,42,2,7830,3,2,0,0,...,2,1,60,0,0,0,3,2,1,2026-06-12 17:30:05.567898
74494,47175,30,0,15,0,3856,2,2,0,2,...,0,1,20,0,0,0,2,2,0,2026-06-12 17:30:05.567898
74495,12409,52,1,5,0,5654,2,3,1,0,...,1,2,7,0,0,0,2,0,0,2026-06-12 17:30:05.567898
74496,9554,18,1,4,0,5276,1,0,0,0,...,1,0,5,0,0,0,3,0,1,2026-06-12 17:30:05.567898


In [5]:
# Step 2 — from this one base, create two versions
employee_df_tree = employee_df_encoded.copy()        # RF, XGBoost — done

In [11]:
employee_df_tree

,employee_id,age,gender,years_at_company,job_role,monthly_income,work-life_balance,job_satisfaction,performance_rating,number_of_promotions,...,job_level,company_size,company_tenure,remote_work,leadership_opportunities,innovation_opportunities,company_reputation,employee_recognition,attrition,event_timestamp
0,8410,31,1,19,0,5390,0,2,0,2,...,1,1,89,0,0,0,0,2,1,2026-06-12 17:30:05.567898
1,64756,59,0,4,3,5534,3,0,3,3,...,1,1,21,0,0,0,1,1,1,2026-06-12 17:30:05.567898
2,30257,24,0,10,2,8159,2,0,3,0,...,1,1,74,0,0,0,3,1,1,2026-06-12 17:30:05.567898
3,65791,36,0,7,0,3989,2,0,2,1,...,1,2,50,1,0,0,2,2,1,2026-06-12 17:30:05.567898
4,65026,56,1,41,0,4821,1,3,0,0,...,2,1,68,0,0,0,1,2,1,2026-06-12 17:30:05.567898
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74493,16243,56,0,42,2,7830,3,2,0,0,...,2,1,60,0,0,0,3,2,1,2026-06-12 17:30:05.567898
74494,47175,30,0,15,0,3856,2,2,0,2,...,0,1,20,0,0,0,2,2,0,2026-06-12 17:30:05.567898
74495,12409,52,1,5,0,5654,2,3,1,0,...,1,2,7,0,0,0,2,0,0,2026-06-12 17:30:05.567898
74496,9554,18,1,4,0,5276,1,0,0,0,...,1,0,5,0,0,0,3,0,1,2026-06-12 17:30:05.567898


In [6]:
employee_df_linear = pd.get_dummies(                 # LR — fix Marital Status on top
    employee_df_encoded.copy(),
    columns=["marital_status"],
    drop_first=True
)

In [7]:
employee_df_linear

,employee_id,age,gender,years_at_company,job_role,monthly_income,work-life_balance,job_satisfaction,performance_rating,number_of_promotions,...,company_tenure,remote_work,leadership_opportunities,innovation_opportunities,company_reputation,employee_recognition,attrition,event_timestamp,marital_status_1,marital_status_2
0,8410,31,1,19,0,5390,0,2,0,2,...,89,0,0,0,0,2,1,2026-06-12 17:30:05.567898,True,False
1,64756,59,0,4,3,5534,3,0,3,3,...,21,0,0,0,1,1,1,2026-06-12 17:30:05.567898,False,False
2,30257,24,0,10,2,8159,2,0,3,0,...,74,0,0,0,3,1,1,2026-06-12 17:30:05.567898,True,False
3,65791,36,0,7,0,3989,2,0,2,1,...,50,1,0,0,2,2,1,2026-06-12 17:30:05.567898,False,True
4,65026,56,1,41,0,4821,1,3,0,0,...,68,0,0,0,1,2,1,2026-06-12 17:30:05.567898,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74493,16243,56,0,42,2,7830,3,2,0,0,...,60,0,0,0,3,2,1,2026-06-12 17:30:05.567898,False,True
74494,47175,30,0,15,0,3856,2,2,0,2,...,20,0,0,0,2,2,0,2026-06-12 17:30:05.567898,True,False
74495,12409,52,1,5,0,5654,2,3,1,0,...,7,0,0,0,2,0,0,2026-06-12 17:30:05.567898,True,False
74496,9554,18,1,4,0,5276,1,0,0,0,...,5,0,0,0,3,0,1,2026-06-12 17:30:05.567898,False,False


### Prepare train/test splits

In [8]:
def prepare_dataset(df, target_col="attrition", timestamp_col="event_timestamp", test_size=0.2, random_state=42):
    X = df.drop(columns=[target_col])
    if timestamp_col in X.columns:
        X = X.drop(columns=[timestamp_col])
    y = df[target_col]
    if y.dtype == object:
        y = y.map({"Left": 1, "Stayed": 0})
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)


### Then split both datasets:

In [10]:
employee_df_tree

,employee_id,age,gender,years_at_company,job_role,monthly_income,work-life_balance,job_satisfaction,performance_rating,number_of_promotions,...,job_level,company_size,company_tenure,remote_work,leadership_opportunities,innovation_opportunities,company_reputation,employee_recognition,attrition,event_timestamp
0,8410,31,1,19,0,5390,0,2,0,2,...,1,1,89,0,0,0,0,2,1,2026-06-12 17:30:05.567898
1,64756,59,0,4,3,5534,3,0,3,3,...,1,1,21,0,0,0,1,1,1,2026-06-12 17:30:05.567898
2,30257,24,0,10,2,8159,2,0,3,0,...,1,1,74,0,0,0,3,1,1,2026-06-12 17:30:05.567898
3,65791,36,0,7,0,3989,2,0,2,1,...,1,2,50,1,0,0,2,2,1,2026-06-12 17:30:05.567898
4,65026,56,1,41,0,4821,1,3,0,0,...,2,1,68,0,0,0,1,2,1,2026-06-12 17:30:05.567898
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74493,16243,56,0,42,2,7830,3,2,0,0,...,2,1,60,0,0,0,3,2,1,2026-06-12 17:30:05.567898
74494,47175,30,0,15,0,3856,2,2,0,2,...,0,1,20,0,0,0,2,2,0,2026-06-12 17:30:05.567898
74495,12409,52,1,5,0,5654,2,3,1,0,...,1,2,7,0,0,0,2,0,0,2026-06-12 17:30:05.567898
74496,9554,18,1,4,0,5276,1,0,0,0,...,1,0,5,0,0,0,3,0,1,2026-06-12 17:30:05.567898


In [9]:
X_tree_train, X_tree_test, y_tree_train, y_tree_test = prepare_dataset(
    employee_df_tree,
    target_col="attrition",
    test_size=0.2,
    random_state=42
)

X_linear_train, X_linear_test, y_linear_train, y_linear_test = prepare_dataset(
    employee_df_linear,
    target_col="attrition",
    test_size=0.2,
    random_state=42
)

In [64]:
from feast import FeatureStore

store = FeatureStore(repo_path="../../attrition_feature_store/feature_repo")

# Use ALL employee IDs — train/test split happens after retrieval, not before.
# Timestamp must be UTC and after the data's event_timestamp (2026-06-12 17:30:05).
# Pinning to the materialize end boundary keeps all rows in scope.
entity_df = employee_df[["employee_id"]].copy().reset_index(drop=True)
entity_df["event_timestamp"] = pd.Timestamp("2026-06-13T00:00:00", tz="UTC")

feast_features = [
    "attrition_features:age",
    "attrition_features:gender",
    "attrition_features:years_at_company",
    "attrition_features:job_role",
    "attrition_features:monthly_income",
    "attrition_features:work-life_balance",
    "attrition_features:job_satisfaction",
    "attrition_features:performance_rating",
    "attrition_features:number_of_promotions",
    "attrition_features:overtime",
    "attrition_features:distance_from_home",
    "attrition_features:education_level",
    "attrition_features:marital_status",
    "attrition_features:number_of_dependents",
    "attrition_features:job_level",
    "attrition_features:company_size",
    "attrition_features:company_tenure",
    "attrition_features:remote_work",
    "attrition_features:leadership_opportunities",
    "attrition_features:innovation_opportunities",
    "attrition_features:company_reputation",
    "attrition_features:employee_recognition",
]

feast_df = store.get_historical_features(
    entity_df=entity_df,
    features=feast_features,
).to_df()

# attrition is the target — not a Feast feature, so merge it back from source
feast_df = feast_df.merge(
    employee_df[["employee_id", "attrition"]],
    on="employee_id",
    how="left",
)

print(f"Retrieved {len(feast_df)} rows, {feast_df.shape[1]} columns from Feast")
# print(feast_df.head())
feast_df.head()


Retrieved 74498 rows, 25 columns from Feast


,employee_id,event_timestamp,age,gender,years_at_company,job_role,monthly_income,work-life_balance,job_satisfaction,performance_rating,...,number_of_dependents,job_level,company_size,company_tenure,remote_work,leadership_opportunities,innovation_opportunities,company_reputation,employee_recognition,attrition
0,8410,2026-06-13 00:00:00+00:00,31,Male,19,Education,5390,Excellent,Medium,Average,...,0,Mid,Medium,89,No,No,No,Excellent,Medium,Stayed
1,67857,2026-06-13 00:00:00+00:00,48,Male,33,Technology,10699,Fair,High,Average,...,0,Entry,Small,70,No,No,No,Fair,High,Stayed
2,42256,2026-06-13 00:00:00+00:00,24,Female,5,Education,4917,Fair,Medium,Average,...,5,Senior,Small,56,Yes,No,No,Good,Medium,Stayed
3,5625,2026-06-13 00:00:00+00:00,46,Male,7,Technology,9189,Fair,Very High,Average,...,2,Mid,Small,31,No,Yes,No,Good,Medium,Left
4,62630,2026-06-13 00:00:00+00:00,32,Male,10,Technology,10698,Excellent,Medium,High,...,3,Senior,Small,69,No,No,Yes,Fair,Low,Stayed


In [65]:
# Encode Feast-retrieved features (raw strings) then split
feast_encoded = feast_df.drop(columns=["event_timestamp"]).copy()

feast_label_encoders = {}
for col in feast_encoded.select_dtypes(include=["object", "category"]).columns:
    if col == "attrition":
        continue
    le = LabelEncoder()
    feast_encoded[col] = le.fit_transform(feast_encoded[col].astype(str))
    feast_label_encoders[col] = le

le_target = LabelEncoder()
feast_encoded["attrition"] = le_target.fit_transform(feast_encoded["attrition"])

X_feast = feast_encoded.drop(columns=["employee_id", "attrition"])
y_feast = feast_encoded["attrition"]

X_feast_train, X_feast_test, y_feast_train, y_feast_test = train_test_split(
    X_feast, y_feast, test_size=0.2, random_state=42, stratify=y_feast
)

print(f"Train: {X_feast_train.shape}  Test: {X_feast_test.shape}")
print(f"Features: {X_feast_train.columns.tolist()}")


Train: (59598, 22)  Test: (14900, 22)
Features: ['age', 'gender', 'years_at_company', 'job_role', 'monthly_income', 'work-life_balance', 'job_satisfaction', 'performance_rating', 'number_of_promotions', 'overtime', 'distance_from_home', 'education_level', 'marital_status', 'number_of_dependents', 'job_level', 'company_size', 'company_tenure', 'remote_work', 'leadership_opportunities', 'innovation_opportunities', 'company_reputation', 'employee_recognition']


###  Log split metadata to MLflow

In [66]:
def log_split_metadata(dataset_name, X_train, X_test, y_train, y_test, test_size, random_state):
    with mlflow.start_run(run_name=f"{dataset_name}-split"):
        mlflow.set_tag("dataset_type", dataset_name)
        mlflow.log_param("test_size", test_size)
        mlflow.log_param("random_state", random_state)
        mlflow.log_param("n_rows_total", len(X_train) + len(X_test))
        mlflow.log_param("n_train_rows", len(X_train))
        mlflow.log_param("n_test_rows", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("feature_names", ",".join(X_train.columns.astype(str)))
        mlflow.log_param("target_classes", ",".join(map(str, sorted(y_train.unique()))))


### Call it for each dataset

In [67]:
log_split_metadata(
    "tree",
    X_tree_train,
    X_tree_test,
    y_tree_train,
    y_tree_test,
    test_size=0.2,
    random_state=42
)

log_split_metadata(
    "linear",
    X_linear_train,
    X_linear_test,
    y_linear_train,
    y_linear_test,
    test_size=0.2,
    random_state=42
)

### Train and Log a model for each dataset

In [ ]:
def train_and_log_model(run_name, dataset_name, model, params=None, param_grid=None,
                        X_train=None, X_test=None, y_train=None, y_test=None, scoring="f1"):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("dataset_type", dataset_name)
        mlflow.set_tag("model_type", model.__class__.__name__)

        if param_grid:
            grid_search = GridSearchCV(
                estimator=model, param_grid=param_grid,
                cv=5, scoring=scoring, n_jobs=-1, verbose=1,
            )
            grid_search.fit(X_train, y_train)
            model = grid_search.best_estimator_
            mlflow.set_tag("grid_search", "true")
            mlflow.log_params(grid_search.best_params_)
            mlflow.log_metric("grid_search_best_score", grid_search.best_score_)
        else:
            if params:
                model.set_params(**params)
            model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
        }
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, "model")
        return metrics, run.info.run_id


In [69]:
from feast import FeatureStore

def fetch_online_features_for_employee(employee_id, feature_list=None):
    store = FeatureStore(repo_path="../../attrition_feature_store/feature_repo")
    if feature_list is None:
        feature_list = [
            "attrition_features:age",
            "attrition_features:gender",
            "attrition_features:years_at_company",
            "attrition_features:job_role",
            "attrition_features:monthly_income",
            "attrition_features:work-life_balance",
            "attrition_features:job_satisfaction",
            "attrition_features:performance_rating",
            "attrition_features:number_of_promotions",
            "attrition_features:overtime",
            "attrition_features:distance_from_home",
            "attrition_features:education_level",
            "attrition_features:marital_status",
            "attrition_features:number_of_dependents",
            "attrition_features:job_level",
            "attrition_features:company_size",
            "attrition_features:company_tenure",
            "attrition_features:remote_work",
            "attrition_features:leadership_opportunities",
            "attrition_features:innovation_opportunities",
            "attrition_features:company_reputation",
            "attrition_features:employee_recognition",
        ]
    return store.get_online_features(
        features=feature_list,
        entity_rows=[{"employee_id": employee_id}],
    ).to_dict()

# print(fetch_online_features_for_employee(8410))


### Example Usage

In [ ]:
tree_metrics, tree_run_id = train_and_log_model(
    run_name="rf-tree",
    dataset_name="tree",
    model=RandomForestClassifier(random_state=42),
    param_grid={
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20],
        "max_features": ["sqrt", "log2"],
        "criterion": ["gini", "entropy"],
    },
    X_train=X_tree_train, X_test=X_tree_test,
    y_train=y_tree_train, y_test=y_tree_test,
    scoring="roc_auc",
)

linear_metrics, linear_run_id = train_and_log_model(
    run_name="logistic-linear",
    dataset_name="linear",
    model=LogisticRegression(random_state=42, max_iter=1000),
    param_grid={
        "C": [0.01, 0.1, 1.0, 10.0],
        "solver": ["liblinear", "saga"],
        "penalty": ["l2"],
        "max_iter": [1000],
    },
    X_train=X_linear_train, X_test=X_linear_test,
    y_train=y_linear_train, y_test=y_linear_test,
    scoring="f1",
)

print("RF metrics:", tree_metrics)
print("LR metrics:", linear_metrics)


In [ ]:
dt_metrics, dt_run_id = train_and_log_model(
    run_name="decision-tree-tree",
    dataset_name="tree",
    model=DecisionTreeClassifier(random_state=42),
    param_grid={
        "criterion": ["gini", "entropy"],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    },
    X_train=X_tree_train, X_test=X_tree_test,
    y_train=y_tree_train, y_test=y_tree_test,
    scoring="f1",
)
print("Decision Tree metrics:", dt_metrics)


In [ ]:
xgb_metrics, xgb_run_id = train_and_log_model(
    run_name="xgboost-tree",
    dataset_name="tree",
    model=XGBClassifier(eval_metric="logloss", random_state=42),
    param_grid={
        "n_estimators": [100, 150],
        "max_depth": [4, 6],
        "learning_rate": [0.05, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
    },
    X_train=X_tree_train, X_test=X_tree_test,
    y_train=y_tree_train, y_test=y_tree_test,
    scoring="roc_auc",
)
print("XGBoost metrics:", xgb_metrics)


In [ ]:
nn_metrics, nn_run_id = train_and_log_model(
    run_name="mlp-linear",
    dataset_name="linear",
    model=MLPClassifier(random_state=42, max_iter=300),
    param_grid={
        "hidden_layer_sizes": [(64, 32), (128, 64)],
        "activation": ["relu", "tanh"],
        "solver": ["adam"],
        "alpha": [0.0001, 0.001],
        "learning_rate_init": [0.001, 0.01],
        "max_iter": [300],
    },
    X_train=X_linear_train, X_test=X_linear_test,
    y_train=y_linear_train, y_test=y_linear_test,
    scoring="f1",
)
print("Neural Network metrics:", nn_metrics)


### Feast Feature Store — Training

In [ ]:
feast_rf_metrics, feast_run_id = train_and_log_model(
    run_name="rf-feast-features",
    dataset_name="feast",
    model=RandomForestClassifier(random_state=42),
    param_grid={
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20],
        "max_features": ["sqrt", "log2"],
        "criterion": ["gini", "entropy"],
    },
    X_train=X_feast_train, X_test=X_feast_test,
    y_train=y_feast_train, y_test=y_feast_test,
    scoring="roc_auc",
)
print("Feast RF metrics:", feast_rf_metrics)


### Lets save the model in registry

In [18]:
# Register the best model (XGBoost had the highest AUC) into MLflow Model Registry
client = MlflowClient()

xgb_run_id  = "ffceef00f9de4a0d97011e35cbb90631"

registered = mlflow.register_model(
    model_uri=f"runs:/{xgb_run_id}/model",
    name=MODEL_NAME,
)
print(f"Registered '{MODEL_NAME}' version {registered.version} from run {xgb_run_id}")



Successfully registered model 'employee-attrition'.
2026/06/13 00:16:33 WARNING mlflow.tracking._model_registry.fluent: Run with id ffceef00f9de4a0d97011e35cbb90631 has no artifacts at artifact path 'model', registering model based on models:/m-045868810dbf482ebc7da4f8016fba1c instead


Registered 'employee-attrition' version 1 from run ffceef00f9de4a0d97011e35cbb90631


Created version '1' of model 'employee-attrition'.


In [25]:
# Notify Slack — training complete + model registered

xgb_metrics = {'accuracy': 0.7606711409395973, 'f1_score': 0.771878198567042, 'precision': 0.7728670253651038, 'recall': 0.7708918987988755}

notify_run_complete(
    run_name="xgboost-tree",
    metrics=xgb_metrics,
    run_id=xgb_run_id,
    webhook_url=SLACK_URL,
)
notify_model_registered(
    model_name=MODEL_NAME,
    version=registered.version,
    run_name="xgboost-tree",
    webhook_url=SLACK_URL,
)


[notify] Slack notification sent (status 200)
[notify] Slack notification sent (status 200)


200

In [26]:
# Promote model to Staging — run this cell when ready to test
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=registered.version,
    stage="Staging",
)
notify_stage_change(
    model_name=MODEL_NAME,
    version=registered.version,
    old_stage="None",
    new_stage="Staging",
    webhook_url=SLACK_URL,
)
print(f"'{MODEL_NAME}' v{registered.version} → Staging")

# Promote to Production — uncomment when validated in Staging
# client.transition_model_version_stage(name=MODEL_NAME, version=registered.version, stage="Production")
# notify_stage_change(MODEL_NAME, registered.version, "Staging", "Production", webhook_url=SLACK_URL)
# print(f"'{MODEL_NAME}' v{registered.version} → Production")


C:\Users\aarun\AppData\Local\Temp\ipykernel_29068\4203755610.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


[notify] Slack notification sent (status 200)
'employee-attrition' v1 → Staging
